# Preparation
**PREPARATION & PREPROCESSING**

---

**Zweck:** Rohdaten bereinigen (NaN-Handling, Datenfehler-Fix, Encoding, Dtype-Optimierung) und exportieren.
**Input:** `telco_churn.db` (SQLite), Findings aus `01_exploration`.
**Output:** `data/processed/telco_churn_clean.csv` (3333 Zeilen, 18 Spalten) — Basis für `03_analysis`.

## Inhalt

- [Data Import](#data-import)
- [Train Test Split](#train-test-split)
- [Data Cleaning](#data-cleaning)
- [Outlier Handling](#outlier-handling)
- [Feature Engineering](#feature-engineering)
- [Data Export](#data-export)

## Data Import

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sqlalchemy import create_engine, text

DATA_RAW       = Path('../data/raw')
DATA_PROCESSED = Path('../data/processed')
DB_PATH        = DATA_RAW / 'telco_churn.db'

query = """
    SELECT
        ch.account_length,
        ch.international_plan,
        ch.voice_mail_plan,
        ch.number_vmail_messages,
        ch.total_day_minutes,
        ch.total_day_calls,
        ch.total_day_charge,
        ch.total_eve_minutes,
        ch.total_eve_calls,
        ch.total_eve_charge,
        ch.total_night_minutes,
        ch.total_night_calls,
        ch.total_night_charge,
        ch.customer_service_calls,
        ch.churn,
        ch.phone_num,
        ci.city,
        ci.area_code
    FROM churn_data AS ch
    LEFT JOIN cities AS ci
    ON ch.local_area_code = ci.area_code
"""

engine = create_engine(f"sqlite:///{DB_PATH}")
with engine.connect() as conn:
    df_raw = pd.read_sql(text(query), conn)

df_eda = df_raw.copy()
df_eda.shape

(3349, 18)

## Train Test Split

**Bewusst uebersprungen (Known Limitation):** Dieses Projekt bestimmt Schwellenwerte fuer Business-Targeting
(welche Kunden/Staedte kontaktieren) auf dem Gesamtdatensatz. Es evaluiert kein Modell auf
Generalisierungsguete — die logistische Regression in `03_analysis` dient der Grenzwert-Herleitung,
nicht der Vorhersage auf ungesehenen Daten. Ein Split waere hier ohne Nutzen und wuerde nur die
Stichprobe fuer die Schwellenwert-Schaetzung unnoetig verkleinern.

## Data Cleaning

In [2]:
df_impute = df_eda.copy()

# Remove fully empty rows
df_impute.dropna(how='all', inplace=True)

# Fill partially-NaN rows consistently (domain knowledge, no statistical parameter from a split):
# - number_vmail_messages: customers without a voice-mail plan have 0 messages -> 0 doesn't bias
# - total_day_calls / total_night_calls / total_eve_calls: isolated capture gaps -> median
df_impute['number_vmail_messages'] = df_impute['number_vmail_messages'].fillna(0)
df_impute['total_day_calls'] = df_impute['total_day_calls'].fillna(df_impute['total_day_calls'].median())
df_impute['total_night_calls'] = df_impute['total_night_calls'].fillna(df_impute['total_night_calls'].median())
df_impute['total_eve_calls'] = df_impute['total_eve_calls'].fillna(df_impute['total_eve_calls'].median())

# Negative call counts are impossible (see plausibility check in 01_exploration:
# 37 rows with -1/-2 in customer_service_calls) -> clip to 0 instead of dropping rows
neg_before = (df_impute['customer_service_calls'] < 0).sum()
df_impute['customer_service_calls'] = df_impute['customer_service_calls'].clip(lower=0)
print(f"customer_service_calls: {neg_before} negative values clipped to 0")

# yes/no -> 1/0 so it can be used in calculations
bool_map = {'yes': 1, 'no': 0}
df_impute['international_plan'] = df_impute['international_plan'].map(bool_map)
df_impute['voice_mail_plan'] = df_impute['voice_mail_plan'].map(bool_map)

df_impute.isna().sum()

customer_service_calls: 37 negative values clipped to 0


account_length            0
international_plan        0
voice_mail_plan           0
number_vmail_messages     0
total_day_minutes         0
total_day_calls           0
total_day_charge          0
total_eve_minutes         0
total_eve_calls           0
total_eve_charge          0
total_night_minutes       0
total_night_calls         0
total_night_charge        0
customer_service_calls    0
churn                     0
phone_num                 0
city                      0
area_code                 0
dtype: int64

## Outlier Handling

In [3]:
df_clean = df_impute.copy()

# customer_service_calls was already clipped in the cleaning step (data error, not an outlier).
# For the remaining columns: no IQR capping -- minute/charge/call values are plausible
# usage data (see plausibility check in 01_exploration), not capture errors.

# Instead: adjust dtypes to the data dictionary (save memory, correct semantics)
dtypes = {
    "account_length": "int16",
    "international_plan": "category",
    "voice_mail_plan": "category",
    "number_vmail_messages": "int8",
    "total_day_minutes": "float32",
    "total_day_calls": "int16",
    "total_day_charge": "float32",
    "total_eve_minutes": "float32",
    "total_eve_calls": "int16",
    "total_eve_charge": "float32",
    "total_night_minutes": "float32",
    "total_night_calls": "int16",
    "total_night_charge": "float32",
    "customer_service_calls": "int8",
    "phone_num": "int16",
    "churn": "int8",
    "city": "category",
    "area_code": "category",
}

memory_before = df_clean.memory_usage(index=True).sum()
df_clean = df_clean.astype(dtypes)
memory_after = df_clean.memory_usage(index=True).sum()
memory_saved_pct = round(100 - (memory_after * 100) / memory_before, 2)

print(f"Memory: {memory_before/1000:.1f} KB -> {memory_after/1000:.1f} KB ({memory_saved_pct}% saved)")
df_clean.dtypes

Memory: 506.6 KB -> 164.4 KB (67.56% saved)


account_length               int16
international_plan        category
voice_mail_plan           category
number_vmail_messages         int8
total_day_minutes          float32
total_day_calls              int16
total_day_charge           float32
total_eve_minutes          float32
total_eve_calls              int16
total_eve_charge           float32
total_night_minutes        float32
total_night_calls            int16
total_night_charge         float32
customer_service_calls        int8
churn                         int8
phone_num                    int16
city                      category
area_code                 category
dtype: object

## Feature Engineering

In [4]:
# No new features planned -- the three indicator candidates are evaluated directly
# from the cleaned raw columns in 03_analysis (see 00_introduction, Scope).

## Data Export

In [5]:
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(DATA_PROCESSED / 'telco_churn_clean.csv', mode='w', index=False, sep=',', encoding='utf-8', na_rep='NA')
print(f'Exported: {df_clean.shape}')

Exported: (3333, 18)
